## Enhancing Public Safety: A Comprehensive Analysis of Geospatial Shifts in Los Angeles and Prediction of Crime Volatility for LAPD Deployment

### Team
Our team, Treeo, consists of three members:
- Ashley Rauch will act as the POC for the group - ashrauch4
- Madeyln Forster - mgforste
- Brealin Redecker - brealinredecker

### Overview
Our crime reporting dataset reflects all incidents of crime in the City of Los Angeles dating back to 2020, collected by The Los Angeles Police Department (LAPD). Ensuring the safety of the public is a fundamental responsibility of the government. Crime, of all levels, can lead to loss of life, physical injury, and significant economic and social impacts on individuals and communities. While a certain level of crime may be considered inevitable, we aim to uncover structural shifts in crime patterns and build a tool that predicts crime volatility and potential "hotspots". The tool will enable the LAPD to develop evidence-based strategies for deployment of its forces as there is a finite number of patrol units available. 

Most crime prediction models explore static relationships, such as linking demographic data to crime rates. These approaches of predictive policing tend to reinforce overpolicing in areas that already get heavy surveillance. Hence, historical "hotspots" are not enough and dismiss crucial aspects of racial biases, current deployment, and accuracy in crime reporting. Long term, instead of just predicting emerging high-volatility crime areas, we are framing it as a deployment question: where should the LAPD actually field its forces? Shifting our focus slightly was important as currently, areas with less police presence result in fewer crime reportings and vice versa, so a model trained only on reported data will keep sending resources to the same places and neglecting everywhere else. We hope to bring in additional data on historical deployment, demographic information on the area, and/or NYC crime data to perform a comparative analysis. Furthermore, we plan to build in an adjustment that estimated how police presence affects the likelihood that a crime actually gets reported and checks against over-concentration and setting minimum coverage thresholds so no area gets completely ignored.

As briefly mentioned, the primary stakeholder for this project is the LAPD. The LAPD are responsible for public safety, and crime incidents are a major public safety concern as they aim to reduce the number of crimes in the areas they police. Success in this project will make two main differences:
1. Ensure resources don't just get locked in historical patterns.
2. Distinguish table high-crime zones from emergencying high-volatility areas so police forces can be adequately deployed.

Overall, our approach and specific needs have not changed. Our team, though, has done some deeper thinking on how we can incorporate other data to improve our model from just predicting hotspots.

### Data
Link to Data: "https://catalog.data.gov/dataset/crime-data-from-2020-to-present"

Our project will utilize data from 'data.gov' and is a crime data set that gives crime information from 2020-Present in Los Angeles. The data has 28 columns, a combination of string, integer, and float features, with over a million rows of data. As the data is government data, it must be accurate, ensuring it is reliable for model use. Also, according to the Freedom of Information Act, they are legally obligated to disclose information — including crime data and records of misconduct — under public records laws. The data includes meta deta, such as that it was published by the Los Angeles Police Department (LAPD), it is publically available data, and it was last updated on January 2, 2026. Some additional aspects we have noted will impact our model, though, is the data is noisy and the specific times when and where crimes get reported versus logged is not always reliable. Many times they will be inputted into the system at the location of the police station in batches of times - this unknown, while can't be changed, needs to be considered when analyzing the model.

### Preprocessing
Our dataset was large, so transformed the raw administrative logs into a clean dataset that is suitable for predicting geospatial volatility. First, we filtered the dataframe to only include crime codes corresponding to assaults, narrowing our scope to high-impact public safety incidents. Next, we addressed data sparsity by dropping columns with over 80% null values. Specifically, we removed "Crm Cd 2", "Crm Cd 3" and "Crm Cd 4" columns as the data was being captured in the "Crm Cd" and "Crm Cd 1" columns. We also removed "Cross Street" as there was a high null percentage and there are other columns to capture the location. This was done to optimize our model's computational efficiency without losing primary intent. Moreover, we transformed the "Date Rptd", "DATE OCC", and "TIME OCC" columns into standardized temporal objects which would allow us to extract cyclical features in our analysis. To ensure we had high quality data for our spatial grid model, we removed invalid coordinates and any entries with missing spatial metadata. Finally, to prepare categorical features for our models, we implemented One-Hot Encoding for victim demographics, and the code-snippet can be seen below. Our group used this cleaned dataset to explore distributions and patterns.

In [6]:
%%capture
'''
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
vict_sex_encoded = encoder.fit_transform(df_filtered[['Vict Sex']].fillna('Unknown'))

df_one_hot = pd.DataFrame(
    vict_sex_encoded, 
    columns=encoder.get_feature_names_out(['Vict Sex']),
    index=df_filtered.index # Important: keeps the rows aligned!
)
'''

### Exploratory Data Analysis

In [7]:
%%capture
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import OneHotEncoder

# Load in the data
data = pd.read_csv("LA_Assault_Data_2020_Present.csv")

TO DO! EXPLORE DISTRIBUTIONS AND CORRELATION OF NUMERIC COLUMNS

In [ ]:
# TO DO! PYTHON FOR DISTRIBUTIONS AND CORRELATION

NEED TO REWRITE BUT CREATES TEMPORAL DISTRIBUTIONS FINDING SPIKES - AND ENSURE IT RUNS PROPERLY

We explore temporal distributions, specifically times and days of the week for which crimes are most likely to occur. 

- The "Midnight Peak": If your plot shows a massive spike at $0$ (Midnight), be aware that this is a common data entry artifact in police records where the exact time isn't known. However, the steady climb from $6:00$ PM to $11:00$ PM is a real behavioral trend.
- The Weekend Effect: You will likely see that Saturday and Sunday accounts for a disproportionate amount of Aggravated Assaults. This supports your Risk Section regarding resource allocation—static deployment doesn't work if the "volatility" is concentrated in a 48-hour window.

In [ ]:
# 1. Temporal Distribution: Hour of Day
# We extract the hour from our cleaned TIME OCC column
data['Hour'] = data['TIME OCC'].apply(lambda x: x.hour)

plt.figure(figsize=(12, 6))
sns.countplot(data=data, x='Hour', hue='Assault Category', palette='viridis')
plt.title('Distribution of Assaults by Hour of Day (Los Angeles)', fontsize=15)
plt.xlabel('Hour (24-hour clock)', fontsize=12)
plt.ylabel('Incident Count', fontsize=12)
plt.legend(title='Crime Type', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.show()

# 2. Temporal Distribution: Day of the Week
# Extract the day name from DATE OCC
data['Day_of_Week'] = pd.to_datetime(data['DATE OCC']).dt.day_name()
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

plt.figure(figsize=(10, 5))
sns.countplot(data=data, x='Day_of_Week', order=day_order, palette='magma')
plt.title('Assault Volume by Day of the Week', fontsize=15)
plt.xlabel('Day', fontsize=12)
plt.ylabel('Incident Count', fontsize=12)
plt.show()

# 3. Crime Severity Mix (Pie Chart)
plt.figure(figsize=(8, 8))
data['Assault Category'].value_counts().plot.pie(autopct='%1.1f%%', startangle=140, colors=sns.color_palette('pastel'))
plt.title('Composition of Assault Crimes in Dataset', fontsize=15)
plt.ylabel('') # Removes the vertical label for cleanliness
plt.show()

AttributeError: 'str' object has no attribute 'hour'